# Visoria: Behavioral Feature Engineering & Attention Modeling

**Objective**: This notebook details the end-to-end Machine Learning pipeline for the Visoria on-device telemetry engine. We aim to classify student engagement (Attentive vs. Distracted) using non-intrusive edge-computed features (head pose, face bounding boxes, and phone presence).

In this enhanced version, we apply **Feature Engineering** to transform raw geometric coordinates into relational context (e.g., Euclidean distance between face and phone), significantly improving the model's spatial awareness and robustness.

## 1. Environment Setup & Data Loading
First, we import the necessary libraries for data manipulation, machine learning, and explainable AI (XAI).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import shap
from IPython.display import display

# Set plotting style for academic presentation
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load the raw V1 dataset
# Note: In Google Colab, ensure this file is uploaded to your environment
df_raw = pd.read_csv('attention_detection_dataset_v1.csv')

print(f"Raw dataset shape: {df_raw.shape}")
display(df_raw.head())

### Exploratory Data Analysis (EDA)
Let's examine the class distribution to ensure our model isn't biased towards a majority class.

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='label', data=df_raw, palette='viridis')
plt.title('Class Distribution (0: Attentive, 1: Distracted)')
plt.xlabel('Label')
plt.ylabel('Count')

# Add count labels on top of bars
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')
plt.show()

## 2. Feature Engineering
**The Problem:** The raw dataset contains raw bounding box coordinates (`face_x`, `face_w`, `phone_x`). These are problematic because they depend heavily on the webcam's resolution and the user's distance from the camera. A phone sitting on a desk far away might trigger a false positive for distraction just because it is in the frame.

**The Solution:** We will engineer **geometric and relational features**:
1. `face_area` & `phone_area`: Gives the model a proxy for depth and proximity to the camera.
2. `face_phone_dist`: The Euclidean distance between the user's face and their phone. 
3. `phone_near_face`: A high-signal binary threshold. If a phone is detected very close to the face, it is almost certainly a distraction event (texting/reading).

In [ ]:
df = df_raw.copy()

# 1. Compute Areas (Proxy for Proximity/Depth)
df['face_area'] = df['face_w'] * df['face_h']
df['phone_area'] = df['phone_w'] * df['phone_h']

# 2. Compute Relational Distance (Face to Phone)
dist = np.sqrt((df['face_x'] - df['phone_x'])**2 + (df['face_y'] - df['phone_y'])**2)
# If no phone is detected (phone == 0), we set distance to a theoretical maximum (9999)
df['face_phone_dist'] = np.where(df['phone'] == 1, dist, 9999.0)

# 3. Compute High-Signal Binary Threshold
# We define "near" as the phone being within 2x the width of the user's face
df['phone_near_face'] = np.where((df['phone'] == 1) & (df['face_phone_dist'] < df['face_w'] * 2.0), 1.0, 0.0)

# Reorder the target 'label' column to the end for neatness
cols = list(df.columns)
cols.remove('label')
cols.append('label')
df = df[cols]

print("Engineered Features successfully added!")
display(df[['phone', 'face_area', 'face_phone_dist', 'phone_near_face', 'label']].head(10))

## 3. Data Preprocessing
Before feeding the data into a Machine Learning model, we must prepare it:
- **One-Hot Encoding**: Converting categorical features (like `pose`) into numerical binaries.
- **Train/Test Split**: Reserving 20% of our data to validate the model's unseen accuracy.
- **Standardization (Scaling)**: Ensuring all numeric features have a mean of 0 and a standard deviation of 1 so large numbers (like `face_area`) don't overpower small numbers (like `pose_x`).

In [ ]:
# Separate Features (X) and Target (y)
X = df.drop(columns=['label'])
y = df['label']

# One-Hot Encode categorical features
X = pd.get_dummies(X, columns=['pose'])

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize numeric features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Testing set shape: {X_test_scaled.shape}")

## 4. Model Training
We utilize a **Random Forest Classifier**. Random Forests are highly robust, non-linear ensemble models that are less prone to overfitting than single decision trees and naturally rank feature importance.

In [ ]:
# Initialize and train the model
rf_model = RandomForestClassifier(
    n_estimators=150, 
    max_depth=12, 
    random_state=42, 
    class_weight='balanced', 
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)
print("Model training complete!")

## 5. Model Evaluation
We evaluate the model on the unseen 20% test set using standard classification metrics.

In [ ]:
# Generate Predictions
y_pred = rf_model.predict(X_test_scaled)

# 1. Classification Report
print("Classification Report:")
print("-" * 53)
print(classification_report(y_test, y_pred, target_names=['Attentive (0)', 'Distracted (1)']))

accuracy = accuracy_score(y_test, y_pred)
print(f"\nOverall Accuracy: {accuracy * 100:.2f}%")

In [ ]:
# 2. Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Attentive', 'Distracted'],
            yticklabels=['Attentive', 'Distracted'])
plt.title('Confusion Matrix', fontsize=14)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.show()

## 6. Explainable AI (XAI) with SHAP
Black-box models are dangerous in educational settings. We use **SHAP (SHapley Additive exPlanations)** to gain game-theoretic interpretability. This allows us to prove exactly *why* the model made its decisions and verify that our engineered features were useful.

In [ ]:
# Initialize SHAP Explainer
explainer = shap.TreeExplainer(rf_model)

# Calculate SHAP values for the test set
shap_values = explainer.shap_values(X_test_scaled)

# 1. Global Feature Importance (Bar Plot)
# This shows which features had the highest absolute impact on the model's predictions overall.
plt.title("Global Feature Importance (SHAP)", fontsize=14)
shap.summary_plot(shap_values[1], X_test, plot_type='bar', show=False)
plt.show()

In [ ]:
# 2. Directional Impact (Summary Dot Plot)
# This plot is incredibly powerful: it shows how the VALUE of a feature affects the prediction.
# Red dots = High feature value
# Blue dots = Low feature value
# Right of center = Pushes prediction towards "Distracted"
# Left of center = Pushes prediction towards "Attentive"

plt.title("Feature Impact Direction (SHAP)", fontsize=14)
shap.summary_plot(shap_values[1], X_test, show=False)
plt.show()

### Conclusion & Observations
By reviewing the SHAP summary plot, we can clinically prove that our engineered features successfully influenced the model's reasoning logic.
1. **`phone_near_face`**: We can see that when this is high (red), it strongly pushes the model to the right (predicting Distracted). 
2. **`pose_y` (Pitch)**: Large deviations in head pitch (looking sharply down or up) correctly penalize attention.

## 7. Artifact Export
Finally, we save the trained model, the scaler, and the expected column schema so the Electron on-device engine can load them for real-time webcam inference.

In [ ]:
import os
os.makedirs('artifacts', exist_ok=True)

joblib.dump(rf_model, 'artifacts/attention_model.pkl')
joblib.dump(scaler, 'artifacts/attention_scaler.pkl')
joblib.dump(list(X.columns), 'artifacts/attention_columns.pkl')

print("âœ… Pipeline complete. Model artifacts successfully exported for Edge Deployment.")